## 1. Configuração e Leitura da Camada Bronze

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

# Path resolution - use notebook's directory
notebook_path = '/Workspace/Users/pierinicoelho@gmail.com/ifood-case/src'
project_root = os.path.abspath(os.path.join(notebook_path, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config.settings import BRONZE_VOLUME_PATH, TARGET_YEARS, TARGET_MONTHS

print("Extraindo metadados dos arquivos dinamicamente...\n")

schema_dict = {}

for year in TARGET_YEARS:
    for month in TARGET_MONTHS:
        formatted_month = str(month).zfill(2)
        database_path = f"{BRONZE_VOLUME_PATH}/yellow/yellow_tripdata_{year}-{formatted_month}.parquet"
        period_key = f"{year}_{formatted_month}"
        
        try:
            file_schema = spark.read.parquet(database_path).schema
            schema_dict[period_key] = {
                data_field.name: data_field.dataType.simpleString() 
                for data_field in file_schema.fields
            }
        except Exception as e:
            print(f"Erro ao ler o arquivo do período {period_key}. Verifique se o caminho esta correto. Erro: {e}")

df_comparison = pd.DataFrame(schema_dict)
df_comparison = df_comparison.fillna("--- AUSENTE ---")

df_comparison = df_comparison.reset_index()
df_comparison = df_comparison.rename(columns={'index': 'column_name'})

period_columns = [col for col in df_comparison.columns if col != 'column_name']

type_conflicts = df_comparison[df_comparison[period_columns].nunique(axis=1) > 1]

if not type_conflicts.empty:
    print("ATENÇÃO! Foram encontrados conflitos de tipos nas seguintes colunas:")
    display(type_conflicts)
else:
    print("Todos os arquivos têm exatamente o mesmo esquema e tipos de dados!")

print("\n--- Esquema completo de todos os períodos ---")
display(df_comparison)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, TimestampType, StringType

# O nosso Contrato de Dados (Schema Enforcement)
schema_bronze_padronizado = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),      # Forçado para Int (ignora o Double de Jan)
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", IntegerType(), True),           # Forçado para Int
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)            # Padrão minúsculo
])

# Instruímos o Spark a ser "case-insensitive" para ele entender que 
# airport_fee e Airport_fee são a mesma coluna
spark.conf.set("spark.sql.caseSensitive", "false")

# Lemos a base inteira impondo o nosso contrato perfeito
df_raw = (
    spark.read
    .schema(schema_bronze_padronizado)
    .parquet(f"{BRONZE_VOLUME_PATH}/yellow/*.parquet")
)

## 3. Estatísticas Descritivas (Single-Pass)

In [0]:
mandatory_columns = [
    "passenger_count", 
    "trip_distance", 
    "total_amount", 
    "fare_amount", 
    "tip_amount"
]

stats = []
for col in mandatory_columns:
    stats.extend([
        F.min(col).alias(f"{col}_min"),
        F.round(F.mean(col), 2).alias(f"{col}_media"),
        F.round(F.stddev(col), 2).alias(f"{col}_desvio"),
        F.max(col).alias(f"{col}_max")
    ])

df_stats = df_raw.select(*stats)

print("Resumo Estatístico:")
display(df_stats)

## 4. Análise de Qualidade: Valores Nulos

In [0]:
null_expression = [
    F.count(F.when(F.col(c).isNull(), True)).alias(c) 
    for c in df_raw.columns
]

df_nulos = df_raw.select(*null_expression)

print("Total de Registros Nulos por Coluna:")
display(df_nulos)

## 5. Check de Duplicidade de Registros para Dedup

In [0]:
total_uniques = df_raw.dropDuplicates().count()
duplicated = total_rows - total_uniques

duplicated_perc = (duplicated / total_rows) * 100 if total_rows > 0 else 0

print("--- RELATÓRIO DE DUPLICIDADE ---")
print(f"Linhas Totais: {total_rows:,}")
print(f"Linhas Únicas: {total_uniques:,}")
print(f"duplicate:    {duplicated:,} ({duplicated_perc:.4f}%)")

## 6. Correlação e Justificativa de Regras (Silver Layer)
Investigação de anomalia (valores negativos), cruzadas com o método de pagamento para determinar se são ruido ou eventos de negócio (ex: Chargebacks).

In [0]:
df_negatives = df_raw.filter(F.col("total_amount") < 0)

df_negative_analysis = (
    df_negatives
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("qtd_corridas_negativas"),
        F.round(F.min("total_amount"), 2).alias("menor_valor_estornado")
    )
    .orderBy(F.desc("qtd_corridas_negativas"))
)

print("Análise de Valores Negativos por Tipo de Pagamento:")
display(df_negative_analysis)

In [0]:
%sql

drop table workspace.ifood_taxi_case.gold_ifood_metrics